# linalg-solve-batched composite — cx10: broadcast a single triangle across rays, then batched solve

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `einops-repeat`, `linalg-solve-batched`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "linalg-solve-batched"
DD_ATOM_IDS = ["einops-repeat", "linalg-solve-batched"]
DD_SUBTOPICS = ["Einops: Repeat", "PyTorch: Batched linalg.solve"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Single-triangle ray intersection over NR rays is a great warm-up for the full batched solver. The setup: one triangle (A, B, C), many rays. For each ray you solve a 3x3 system `[-D, B-A, C-A] @ [s, u, v]^T = O - A`. Vectorize across NR rays with `torch.linalg.solve`.

Atoms compose like this:
  1. `einops-repeat` broadcasts the single triangle's `(B-A)` and `(C-A)` edge vectors from `(3,)` to `(NR, 3)` so the LHS can be built per-ray.
  2. `linalg-solve-batched` runs `torch.linalg.solve(LHS, RHS)` on the resulting `(NR, 3, 3)` and `(NR, 3)` stacks in one shot — the batch axis is leading.

The reason we don't just write a Python loop: solve-batched dispatches to a fused LAPACK kernel.

### Composite Exercise — broadcast a single triangle across rays, then batched solve

**Atoms exercised together**: `einops-repeat`, `linalg-solve-batched`

Implement `cx10_intersect_rays_one_triangle(rays, A, B, C)` — for each of `NR` rays, solve the ray-triangle system for `(s, u, v)`. Ray `i` is `rays[i] = [origin_i, direction_i]` of shape `(2, 3)`.

The system per-ray is
  `[-D_i | B-A | C-A] @ [s_i, u_i, v_i]^T = O_i - A`.

1. Compute edge vectors `e1 = B - A` and `e2 = C - A` — each `(3,)`.
2. **Repeat** them to `(NR, 3)`: `repeat(e1, 'd -> r d', r=NR)` and similarly for `e2`. Stride-0 views — no copies.
3. Negate the per-ray directions, stack `[-D, e1_b, e2_b]` along a new axis to form the `(NR, 3, 3)` LHS (column order matters: column 0 = -D, column 1 = e1, column 2 = e2). Hint: stack along `dim=-1` so columns line up.
4. Compute the RHS `O - A` of shape `(NR, 3)`.
5. **linalg-solve-batched**: `t.linalg.solve(LHS, RHS)` returns `(NR, 3)` — the per-ray `(s, u, v)`.

Cross-check by reconstructing: for each ray, `O + s*D` should equal `A + u*e1 + v*e2`.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx10_intersect_rays_one_triangle(rays, A, B, C):
    raise NotImplementedError

def _test_cx10():
    # Case A: hand-built rays that all hit the same triangle.
    A = t.tensor([0.0, 0.0, 0.0])
    B = t.tensor([1.0, 0.0, 0.0])
    C = t.tensor([0.0, 1.0, 0.0])
    # Three rays, all from z=1 pointing down (-z), aimed at known triangle points.
    origins = t.tensor([[0.25, 0.25, 1.0], [0.5, 0.1, 1.0], [0.1, 0.5, 1.0]])
    dirs = t.tensor([[0.0, 0.0, -1.0]] * 3)
    rays = t.stack([origins, dirs], dim=1)  # (3, 2, 3)
    out = cx10_intersect_rays_one_triangle(rays, A, B, C)
    assert tuple(out.shape) == (3, 3), f'shape: {tuple(out.shape)}'
    s, u, v = out[:, 0], out[:, 1], out[:, 2]
    # All rays travel a distance of 1 along -z.
    assert t.allclose(s, t.ones(3), atol=1e-5), f's: {s}'
    # (u, v) should equal the (x, y) of each origin since triangle is at origin with edges along x and y.
    assert t.allclose(u, origins[:, 0], atol=1e-5), f'u: {u}'
    assert t.allclose(v, origins[:, 1], atol=1e-5), f'v: {v}'

    # Case B: reconstruction round-trip — random rays + non-degenerate triangle.
    t.manual_seed(42)
    A2 = t.randn(3)
    B2 = A2 + t.tensor([1.0, 0.0, 0.0])
    C2 = A2 + t.tensor([0.0, 1.0, 0.0])
    rays2 = t.randn(20, 2, 3)
    # Make rays point somewhere consistent.
    rays2[:, 1] = t.tensor([0.0, 0.0, -1.0])
    out2 = cx10_intersect_rays_one_triangle(rays2, A2, B2, C2)
    assert tuple(out2.shape) == (20, 3)
    s2, u2, v2 = out2[:, 0:1], out2[:, 1:2], out2[:, 2:3]
    lhs = rays2[:, 0] + s2 * rays2[:, 1]  # O + s*D
    e1_2 = B2 - A2
    e2_2 = C2 - A2
    rhs = A2 + u2 * e1_2 + v2 * e2_2
    assert t.allclose(lhs, rhs, atol=1e-4), f'round-trip failed: max diff {(lhs-rhs).abs().max()}'
    _dd_passed.add('cx10')

_test_cx10()

<details><summary>Show solution — cx10</summary>

```python
def cx10_intersect_rays_one_triangle(rays, A, B, C):
    NR = rays.shape[0]
    O = rays[:, 0]   # (NR, 3)
    D = rays[:, 1]   # (NR, 3)
    e1 = B - A       # (3,)
    e2 = C - A       # (3,)
    # Atom A (einops-repeat): broadcast (3,) edges to (NR, 3) — stride-0 views.
    e1_b = repeat(e1, 'd -> r d', r=NR)
    e2_b = repeat(e2, 'd -> r d', r=NR)
    # Build LHS: columns [-D, e1_b, e2_b] — stack along last dim so each column is a 3-vec.
    LHS = t.stack([-D, e1_b, e2_b], dim=-1)  # (NR, 3, 3)
    RHS = O - A                                # (NR, 3)
    # Atom B (linalg-solve-batched): solve all NR systems in one fused call.
    return t.linalg.solve(LHS, RHS)
```

`torch.linalg.solve(A, b)` accepts a leading batch axis: `A` shape `(*, n, n)`, `b` shape `(*, n)`. Under the hood it dispatches to a batched LAPACK gesv, which is dramatically faster than a Python `for` loop over single solves. The einops repeats are zero-copy, so the dominant cost is the solve itself — exactly what you want.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx10'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx10',
        'subtopics': ["Einops: Repeat", "PyTorch: Batched linalg.solve"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()